https://automatetheboringstuff.com/3e/chapter20.html

Enabling the API
Before you write code, you must first sign up for a Gmail email account at https://gmail.com. Then, you must set up the Gmail API for your account through the Google Cloud console at https://console.cloud.google.com. These steps are identical to the steps for setting up EZSheets detailed in Chapter 15, so I won’t repeat them in this chapter. Create a new project and make sure to enable the Gmail API instead of the Google Sheets API. On step 2 of the OAuth consent screen configuration, add the https://mail.google.com scope to let your Python scripts read and send email. When you’re done, you should have a credentials file and a token file.

Then, in the interactive shell, enter the following code:

In [1]:
import ezgmail
ezgmail.init()

ModuleNotFoundError: No module named 'ezgmail'

If no error appears, EZGmail has been correctly installed.

Sending Mail
Once EZGmail is configured, you should be able to send email with a single function call:

In [ ]:
import ezgmail
ezgmail.send('recipient@example.com', 'Subject line', 'Body of the email')

In [ ]:
ezgmail.send('recipient@example.com', 'Subject line', 'Body of the email',
['attachment1.jpg', 'attachment2.mp3'])

In [ ]:
import ezgmail
ezgmail.send('recipient@example.com', 'Subject line', 'Body of the email', cc='friend@example.com', bcc='otherfriend@example.com,someoneelse@example.com')

In [ ]:
# >>> import ezgmail
# >>> ezgmail.EMAIL_ADDRESS
# 'example@gmail.com'

Reading Mail
Gmail organizes email messages that are replies to each other into conversation threads. When you log in to Gmail in your web browser or through an app, you’re really looking at email threads rather than individual emails (even if the thread has only one email in it).

EZGmail has GmailThread and GmailMessage objects to represent conversation threads and individual emails, respectively. A GmailThread object has a messages attribute that holds a list of GmailMessage objects. The unread() function returns a list of GmailThread objects for the 25 most recent unread emails, which can then be passed to ezgmail.summary() to print a summary of the conversation threads in that list:

In [ ]:
# >>> import ezgmail
# >>> unread_threads = ezgmail.unread()  # List of GmailThread objects
# >>> ezgmail.summary(unread_threads)
# Al, Jon - Do you want to watch RoboCop this weekend? - Dec 09
# Jon - Thanks for stopping me from buying Bitcoin. - Dec 09

In [ ]:
# >>> len(unread_threads)
# 2
# >>> str(unread_threads[0])
# "<GmailThread len=2 snippet= Do you want to watch RoboCop this weekend?'>"
# >>> len(unread_threads[0].messages)
# 2
# >>> str(unread_threads[0].messages[0])
# "<GmailMessage from='Al Sweigart <al@inventwithpython.com>' to='Jon Doe
# <example@gmail.com>' timestamp=datetime.datetime(2026, 12, 9, 13, 28, 48)
# subject='RoboCop' snippet='Do you want to watch RoboCop this weekend?'>"
# >>> unread_threads[0].messages[0].subject
# 'RoboCop'
# >>> unread_threads[0].messages[0].body
# 'Do you want to watch RoboCop this weekend?\r\n'
# >>> unread_threads[0].messages[0].timestamp
# datetime.datetime(2026, 12, 9, 13, 28, 48)
# >>> unread_threads[0].messages[0].sender
# 'Al Sweigart <al@inventwithpython.com>'
# >>> unread_threads[0].messages[0].recipient
# 'Jon Doe <example@gmail.com>'

In [ ]:
# >>> recent_threads = ezgmail.recent()
# >>> len(recent_threads)
# 25
# >>> recent_threads = ezgmail.recent(maxResults=100)
# >>> len(recent_threads)
# 46

Searching for Mail
In addition to using ezgmail.unread() and ezgmail.recent(), you can search for specific emails, the same way you would if you entered queries into the Gmail search box, by calling ezgmail.search():

In [ ]:
# >>> result_threads = ezgmail.search('RoboCop')
# >>> len(result_threads)
# 1
# >>> ezgmail.summary(result_threads)
# Al, Jon - Do you want to watch RoboCop this weekend? - Dec 09

Downloading Attachments
A GmailMessage object has an attachments attribute that is a list of filenames for the message’s attached files. You can pass any of these names to a GmailMessage object’s downloadAttachment() method to download the files. You can also download all of them at once with downloadAllAttachments(). By default, EZGmail saves attachments to the current working directory, but you can pass an additional downloadFolder keyword argument to downloadAttachment() and downloadAllAttachments() as well. Here is an example:

In [ ]:
# >>> import ezgmail
# >>> threads = ezgmail.search('vacation photos')
# >>> threads[0].messages[0].attachments
# ['tulips.jpg', 'canal.jpg', 'bicycles.jpg']
# >>> threads[0].messages[0].downloadAttachment('tulips.jpg')
# >>> threads[0].messages[0].downloadAllAttachments(downloadFolder='vacation2026')
# ['tulips.jpg', 'canal.jpg', 'bicycles.jpg']

Sending Notifications
Sending a push notification to everyone who is subscribed to a topic requires nothing more than making an HTTP request to the ntfy web server. This means it can be done entirely with the Requests library. You don’t need to install a ntfy-specific package.

To send the request, enter the following into the interactive shell. Replace the AlSweigartZPgxBQ42 example topic used throughout this chapter with your own random, secret topic:

In [ ]:
# >>> import requests
# >>> requests.post('https://ntfy.sh/AlSweigartZPgxBQ42', 'Hello, world!')
# <Response [200]>

<!-- Transmitting Metadata -->

In [ ]:
# >>> import requests
# >>> requests.post('https://ntfy.sh/AlSweigartZPgxBQ42', 'The rent is too high!', 
# headers={'Title':'Important: Read this!', 'Tags': 'warning,neutral_face', 'Priority':'5'})
# <Response [200]>

Receiving Notifications
Your Python programs can also read the messages posted to a particular topic by making HTTP requests with the Requests library. Send a notification message using the code in the previous sections, and then enter the following into the interactive shell using the same topic as the notifications:

In [ ]:
# >>> import requests
# >>> resp = requests.get('https://ntfy.sh/AlSweigartZPgxBQ42/json?poll=1')
# >>> resp.text
# '{"id":"1jnHKeFNqwnS","time":1797823340,"expires":1797866540,"event":
# "message","topic":"AlSweigartZPgxBQ42","message":"Hello, world!"}\n
# {"id":"wZ22cjyKXw1F","time":1797823712,"expires":1797866912,"event":
# "message","topic":"AlSweigartZPgxBQ42","title":"Important: Read this!",
# "message":"The rent is too high!","priority":5,"tags":["warning",
# "neutral_face"]}\n'

In [ ]:
# >>> import json
# >>> notifications = []
# >>> for json_text in resp.text.splitlines():
# ...     notifications.append(json.loads(json_text))
# ...
# >>> notifications[0]['message']
# 'Hello, world!'
# >>> notifications[1]['message']
# 'The rent is too high!'

Email-Based Computer Control

In [ ]:
# qbProcess = subprocess.Popen(['C:\\Program Files (x86)\\qBittorrent\\
# qbittorrent.exe', 'shakespeare_complete_works.torrent'])